In [ ]:
# 복수의 LLM 요청을 비동기 병렬로 처리
# 웹크롤링(3개) -> LLM 요약 -> chroma에 저장 -> RAG 검색 -> LLM 최종 답변

!pip install openai sentence-transformers chromadb python-dotenv
!pip install request beautifulsoup4

In [ ]:
from openai import OpenAI
from openai import AsyncOpenAI
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb import PersistentClient
from bs4 import BeautifulSoup
import requests
import textwrap   # 긴 문자열을 보기좋게 줄바꿈 가능
import os
import asyncio
from dotenv import load_dotenv

# async def hello():
#     print('시작')
#     await asyncio.sleep(2)
#     print('2초 후 실행')

# await hello()

load_dotenv()

URLS = [
    'https://ko.wikipedia.org/wiki/김치찌게',
    'https://ko.wikipedia.org/wiki/인공지능',
    'https://ko.wikipedia.org/wiki/야구',
]
GPT_MODEL = "gpt-4o-mini"
CHROMA_PATH = "./wiki_async"
COLLECTION_NAME = "wiki_docs"
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise RuntimeError("OPENAI_API_KEY가 없어요")

# 동기 방식
client = OpenAI(api_key=OPENAI_API_KEY)

# 비동기 방식
async_client = AsyncOpenAI(api_key=OPENAI_API_KEY)

embedder = SentenceTransformer("all-MiniLM-L6-v2")
chroma = PersistentClient(path=CHROMA_PATH)
collection = chroma.get_or_create_collection(COLLECTION_NAME)


In [ ]:
# 위키백과 URL에서 문서 읽기 함수
def fetch_wiki(url:str, max_chars:int=3000) -> str:
    headers={'USer-Agent':'Mozilla/5.0'}
    response = requests.get(url, headers=headers)
    print(f'[FETCH] {url} -> {response.status_code}')

    if response.status_code != 200:
        return ""

    soup = BeautifulSoup(response.text, 'html.parser')
    paragraphs = [
        p.get_text(strip=True) for p in soup.find_all('p') if p.get_text(strip=True)
    ]
    text = '\n'.join(paragraphs)
    return text[:max_chars]


# LLM에게 비동기로 요약을 요청하는 함수
async def summarize_async(url:str, text:str) -> str:
    if not text:
        return ""

    prompt = f"""
    너는 한국어 문서를 잘 요약하는 전문가야.

    아래 문서를 핵심 개념, 중요 사실, 특징 중심으로 10 문장 이내로 요약해 줘.

    URL:
    {url}

    문서 내용:
    {text}
    """

    response = await async_client.responses.create(
        model = GPT_MODEL,
        input = prompt,
        temperature=0
    )
    return response.output_text.strip()


# chromadb에 저장을 위한 비동기 함수
async def build_db_async():
    texts = []   # URL과 본문 텍스트를 저장

    for url in URLS:
        text = fetch_wiki(url)
        texts.append((url, text))

    tasks = [
        summarize_async(url, text) for url, text in texts
    ]

    # 여러 LLM 요약 요청을 동시에 실행하고 그 결과를 리스트로 받기
    summarizes = await asyncio.gather(*tasks)

    docs = []   # chromadb에 저장용
    ids = []
    metadatas = []

    for i, ((url, _), summary) in enumerate(zip(texts, summarizes)):
        if not summary:
            continue

        docs.append(summary)
        ids.append(f'doc_{i}')
        metadatas.append({'url':url})

    if not docs:
        print('[알림] 저장할 문서가 없어요')
        return

    embeddings = embedder.encode(docs).tolist()

    collection.upsert(
        ids=ids,
        documents=docs,
        embeddings=embeddings,
        metadatas=metadatas
    )
    print(f'벡터디비에 저장 완료. 현재 문서 수 : {collection.count()}')

    print('요약 결과:')
    for meta, doc in zip(metadatas, docs):
        print('=' * 20)
        print('URL : ', meta['url'])
        print(textwrap.fill(doc, width=70))



In [ ]:
# RAG 검색 + LLM 최종 답변
def answer_with_rag(query:str, top_k:int=3):
    print('\n' + '***' * 30)
    print('[질문]', query)

    q_vec = embedder.encode([query])[0].tolist()

    # chroma에서 질문과 유사한 문서 검색
    result = collection.query(
        query_embeddings=[q_vec],
        n_results=top_k,
        include=['documents', 'metadatas', 'distances']
    )

    docs = result['documents'][0]
    metas = result['metadatas'][0]
    dists = result['distances'][0]

    context_blocks = []  # LLM에게 넘길 컨텍스트 블록

    print('\n[검색 결과]')
    for i, (doc, meta, dist) in enumerate(zip(docs, metas, dists), start=1):
        url = meta['url']  # 문서 출처 url
        print('-' * 50)
        print(f'{i}, URL:{url}')
        print(f'  distance:{dist:.4f}')
        print(textwrap.fill(doc, width=70))

        context_blocks.append(f'[{i}] 출처 : {url}\n{doc}')

    context = '\n\n'.join(context_blocks)

    # LLM에게 보낼 최종 RAG 프롬프트
    prompt = f"""
    너는 RAG 기반 한국어 전문 도우미야.
    아래 검색된 문서 유약을 근거로 사용자의 질문에 답을 해줘.
    컨텍스트에 없는 내용은 지어내지 말고 말해,

    사용자 질문
    {query}

    검색된 컨텍스트
    {context}

    답변:
    """

    response = client.responses.create(
        model = GPT_MODEL,
        input=prompt,
        temperature=0.3
    )
    print('\n최종 답변')
    print(textwrap.fill(response.output_text.strip(), width=70))

# 실행
await build_db_async()  # 웹 문서를 읽어 LLM에게 요약하게 한 뒤 vectordb에 저장하는 작업을 비동기 처리

questions = [
    '김치찌게의 특징과 재료를 설명해 줘',
    '인공지능이 뭐니?',
    '야구는 어떤 스포츠야?'
]

for q in questions:
    answer_with_rag(q)
